# Junior Data Engineering Interview Solutions

### Prerequisites
We will install `duckdb` to run production-grade SQL (including Window functions and `MERGE` statements) directly on Python data structures without needing a heavy database setup.

In [1]:
!pip install duckdb pandas numpy --quiet
import pandas as pd
import numpy as np
import duckdb
print("Setup complete! Ready to execute scenarios.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup complete! Ready to execute scenarios.


## Scenario 1: SQL Data Deduplication
**Goal:** Keep only the most recent click event for each unique user from a raw dataset with duplicate entries using `ROW_NUMBER()`.

In [2]:
# 1. Create a dummy dataframe representing a messy raw table
raw_clicks_data = pd.DataFrame([
    {"user_id": "USR_A", "click_timestamp": "2026-03-01 10:00:00", "page_id": "home_page"},
    {"user_id": "USR_B", "click_timestamp": "2026-03-01 10:05:00", "page_id": "cart_page"},
    {"user_id": "USR_A", "click_timestamp": "2026-03-01 10:15:00", "page_id": "checkout_page"},  # Most recent for A
    {"user_id": "USR_B", "click_timestamp": "2026-03-01 10:02:00", "page_id": "pricing_page"}
])

# Register DataFrame with DuckDB so we can run pure SQL on it
conn = duckdb.connect()
conn.register("raw_clicks", raw_clicks_data)

# 2. Run the production SQL solution
dedup_sql = """
WITH RankedUserClicks AS (
    SELECT
        user_id,
        click_timestamp,
        page_id,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY click_timestamp DESC
        ) as row_num
    FROM raw_clicks
)
SELECT
    user_id,
    click_timestamp,
    page_id
FROM RankedUserClicks
WHERE row_num = 1;
"""

print("--- Scenario 1 Results (Deduplicated Clicks) ---")
print(conn.execute(dedup_sql).df())

--- Scenario 1 Results (Deduplicated Clicks) ---
  user_id      click_timestamp        page_id
0   USR_B  2026-03-01 10:05:00      cart_page
1   USR_A  2026-03-01 10:15:00  checkout_page


## Scenario 2: Python Data Cleaning (REST API to Pandas)
**Goal:** Clean a list of dictionaries from a REST API containing missing/`None` values before converting it into a structured DataFrame.

In [3]:
# Mock raw API payload containing missing values
raw_api_response = [
    {"item_id": 101, "item_name": "Laptop", "price": 1200.00},
    {"item_id": 102, "item_name": "Wireless Mouse", "price": None},
    {"item_id": 103, "item_name": "4K Monitor", "price": 350.00},
    {"item_id": 104, "item_name": "Mechanical Keyboard", "price": None}
]

# List comprehension fallback data cleaning approach
cleaned_records = [
    {
        "item_id": record["item_id"],
        "item_name": record["item_name"],
        "price": float(record["price"]) if record["price"] is not None else 0.0
    }
    for record in raw_api_response
]

# Convert into a DataFrame
df_cleaned = pd.DataFrame(cleaned_records)

print("--- Scenario 2 Results (Cleaned API Data Frame) ---")
print(df_cleaned)

--- Scenario 2 Results (Cleaned API Data Frame) ---
   item_id            item_name   price
0      101               Laptop  1200.0
1      102       Wireless Mouse     0.0
2      103           4K Monitor   350.0
3      104  Mechanical Keyboard     0.0


## Scenario 3: Implementing Pipeline Idempotency
**Goal:** Ensure that running your pipeline task multiple times does not result in duplicate records by using an atomic `MERGE` statement.

In [4]:
# Setting up our warehouse state table
target_users = pd.DataFrame([
    {"user_id": 1, "email": "alice@example.com", "last_active": "2026-01-01"},
    {"user_id": 2, "email": "bob@example.com", "last_active": "2026-01-01"}
])

# New batch incoming from a daily pipeline run (Alice updated her email, Charlie is a new sign up)
daily_source_users = pd.DataFrame([
    {"user_id": 1, "email": "alice_new@example.com", "last_active": "2026-03-01"},
    {"user_id": 3, "email": "charlie@example.com", "last_active": "2026-03-01"}
])

conn.execute("DROP TABLE IF EXISTS warehouse_users;")
conn.execute("CREATE TABLE warehouse_users AS SELECT * FROM target_users")
conn.register("staging_users", daily_source_users)

# Idempotent Merge SQL Query (replaced with UPDATE and INSERT for broader compatibility)
# First, update existing records
update_sql = """
UPDATE warehouse_users AS target
SET
  email = source.email,
  last_active = source.last_active
FROM staging_users AS source
WHERE target.user_id = source.user_id;
"""
conn.execute(update_sql)

# Second, insert new records
insert_sql = """
INSERT INTO warehouse_users (user_id, email, last_active)
SELECT
  source.user_id,
  source.email,
  source.last_active
FROM staging_users AS source
LEFT JOIN warehouse_users AS target
  ON source.user_id = target.user_id
WHERE target.user_id IS NULL;
"""
conn.execute(insert_sql)

print("--- Scenario 3 Results (Idempotent Warehouse State After Merge) ---")
print(conn.execute("SELECT * FROM warehouse_users").df())

--- Scenario 3 Results (Idempotent Warehouse State After Merge) ---
   user_id                  email last_active
0        1  alice_new@example.com  2026-03-01
1        2        bob@example.com  2026-01-01
2        3    charlie@example.com  2026-03-01


## Scenario 4: OOM (Out Of Memory) Prevention
**Goal:** Build a chunked processing data loop using pandas to incrementally stream a file to disk instead of filling up the entire server memory profile.

In [5]:
# Generate a mock dataset representing a giant log file
huge_data_mock = pd.DataFrame({
    "transaction_id": range(1000, 1010),
    "amount": [10.5, 99.0, 400.0, 5.25, 12.0, 85.0, 120.0, 34.0, 56.0, 11.0]
})
huge_data_mock.to_csv("large_source_file.csv", index=False)

print("Mock file 'large_source_file.csv' written to local workspace disk.\n")

# Low memory chunking extraction loop
chunk_size = 3  # Low number chosen to visually showcase chunk loops executing
db_engine = duckdb.connect("production_warehouse.db")

print("Starting memory-safe streaming transaction execution loop...")
for i, chunk in enumerate(pd.read_csv("large_source_file.csv", chunksize=chunk_size)):
    # Add transformation step flags per batch chunk
    chunk['processed_at_chunk_id'] = i

    # Append to database
    if i == 0:
        db_engine.register("first_chunk", chunk)
        db_engine.execute("CREATE TABLE target_transactions AS SELECT * FROM first_chunk")
    else:
        db_engine.register(f"chunk_{i}", chunk)
        db_engine.execute(f"INSERT INTO target_transactions SELECT * FROM chunk_{i}")

    print(f" -> Successfully committed batch chunk {i + 1} safely into DB.")

print("\n--- Scenario 4 Results (Final Saved Production DB Table Sample) ---")
print(db_engine.execute("SELECT * FROM target_transactions LIMIT 5").df())

Mock file 'large_source_file.csv' written to local workspace disk.

Starting memory-safe streaming transaction execution loop...
 -> Successfully committed batch chunk 1 safely into DB.
 -> Successfully committed batch chunk 2 safely into DB.
 -> Successfully committed batch chunk 3 safely into DB.
 -> Successfully committed batch chunk 4 safely into DB.

--- Scenario 4 Results (Final Saved Production DB Table Sample) ---
   transaction_id  amount  processed_at_chunk_id
0            1000   10.50                      0
1            1001   99.00                      0
2            1002  400.00                      0
3            1003    5.25                      1
4            1004   12.00                      1
